# 03 — Metrics, branch-fusion ablation, and an honest speedup

**Addresses Blocker 3 plus the secondary items a C&F referee will raise.**

| Experiment | Manuscript target |
|---|---|
| 1. Anomaly-normalised error metrics | §4.2–4.5, Table 3, abstract |
| 2. Conservation diagnostics on the operator | **new** — §4, Fig. W2 |
| 3. Branch-fusion ablation (add / concat / bilinear) | §3.4.2, Table 4 |
| 4. Strong-bump case C2b | §4.3 — tests the fusion claim properly |
| 5. Like-for-like speedup benchmark | §4.9, Table 5 |

Cells 1–2 and 5 run on data alone. Cells 3–4 train models; expect ~15 min each on a T4.

In [ ]:
import numpy as np, time
import matplotlib.pyplot as plt
from swe_solvers import swe_solve, cell_centers, rel_l2, rel_l2_anomaly, G

L, T = 10.0, 1.0
d = np.load("swe_data_wb.npz")            # from notebook 01
xg, H0, BB = d["x"], d["h0"], d["b"]
t_snap, H_snap, HU_snap = d["t_snap"], d["h"], d["hu"]
print("data:", H_snap.shape, "| generation cost", float(d["gen_seconds"]), "s")

## 1. Error metrics that are not flattered by the background depth

Report all three. The first is what the paper currently reports; the second is what a
hydraulics referee considers meaningful; the third is unambiguous.

In [ ]:
def error_triplet(pred, ref, h_rest=None):
    r = ref.mean() if h_rest is None else h_rest
    return dict(rel_total=rel_l2(pred, ref),
                rel_anomaly=rel_l2_anomaly(pred, ref, r),
                rmse_m=float(np.sqrt(np.mean((pred - ref) ** 2))))

# inflation factor of the current metric, per snapshot time
print(f"{'t [s]':>7}{'||h||/||h-hbar||':>20}")
for i, t_ in enumerate(t_snap):
    f = H_snap[:, i, :]
    infl = np.linalg.norm(f, axis=1) / np.linalg.norm(f - f.mean(axis=1, keepdims=True), axis=1)
    print(f"{t_:>7.2f}{infl.mean():>20.1f}")
print("\nMultiply any reported rel_total by these to get the error on the wave signal.")

# demo on a synthetic 'prediction' = reference + smooth perturbation
ref = H_snap[0, -1, :]
pred = ref + 0.01 * np.sin(6 * np.pi * xg / L)
print("\nexample:", {k: f"{v:.3e}" for k, v in error_triplet(pred, ref, 1.0).items()})

**Recommended reporting for Table 3.** Keep `rel_total` for continuity with the
literature, add `rel_anomaly` and `rmse_m` as the primary columns, and state the
normalisation explicitly in the caption. Also add $\bar\varepsilon_{hu} = 2.28\times10^{-1}$
to the abstract — omitting it while quoting the depth figure reads as selective.

## 2. Conservation diagnostics on the operator prediction

Plug your trained model in below. Neural operators typically violate mass
conservation at the percent level; showing that you measured it is worth more than a
good number, and hiding it is not an option at C&F.

In [ ]:
def operator_conservation(predict_fn, h0, b, x, times):
    """
    predict_fn(h0, b, x, t) -> (h, hu) on the grid x at scalar time t.
    Returns relative mass drift and total-momentum history.
    """
    dx = x[1] - x[0]
    M0 = np.sum(h0) * dx
    mass, mom = [], []
    for t_ in times:
        h, hu = predict_fn(h0, b, x, t_)
        mass.append(abs(np.sum(h) * dx - M0) / M0)
        mom.append(np.sum(hu) * dx)
    return np.array(mass), np.array(mom)

# reference behaviour for comparison (flat bed => momentum is conserved too)
times = np.linspace(0.05, T, 20)
_, out = swe_solve(xg, H0[0], BB[0], T, cfl=0.45, order=2,
                   snapshots=[float(t_) for t_ in times])
dx = xg[1] - xg[0]; M0 = np.sum(H0[0]) * dx
ref_mass = np.array([abs(np.sum(out[float(t_)][0]) * dx - M0) / M0 for t_ in times])
print("reference solver final relative mass drift:", f"{ref_mass[-1]:.3e}")

# --- plug in your model, then uncomment ---
# op_mass, op_mom = operator_conservation(my_predict, H0[0], BB[0], xg, times)
# plt.semilogy(times, ref_mass + 1e-18, 'k-', label='WB-HLL reference')
# plt.semilogy(times, op_mass + 1e-18, 'r-', label='DeepONet')
# plt.xlabel('t [s]'); plt.ylabel('|ΔM|/M₀'); plt.legend(); plt.show()

## 3. Branch-fusion ablation

§3.4.2 says the $h_0$–$b$ interaction "is instead mediated through the shared trunk".
The trunks take only $(x,t)$ and never see $h_0$ or $b$, so they cannot mediate anything.
With $\boldsymbol\beta = B_1(h_0) + B_2(b)$ the coefficient map is **additively separable**,
while the source term $-gh\,\partial_x b$ is bilinear.

C2 does not refute this because $b$ has amplitude 0.12 m — the coupling is weak.
Train three fusion variants and evaluate on both C2 and a strong-bump C2b.

In [ ]:
import tensorflow as tf
from deeponet_tf import FourBranchDeepONet

M = 100
xs = np.linspace(0.0, L, M, endpoint=False).astype(np.float32)
NXQ = H_snap.shape[-1]
N_SUP = int(d["n_sup"])

def grid_interp_np(f, x_src, x_q):
    return np.interp(x_q, x_src, f, period=L)

def make_dataset():
    h0s = np.stack([grid_interp_np(H0[j], xg, xs) for j in range(N_SUP)]).astype(np.float32)
    bs  = np.stack([grid_interp_np(BB[j], xg, xs) for j in range(N_SUP)]).astype(np.float32)
    return h0s, bs

H0S, BS = make_dataset()
XQ = xg.astype(np.float32)
TS = t_snap.astype(np.float32)
print("branch inputs:", H0S.shape, BS.shape)

In [ ]:
def train_variant(fusion, steps=15000, lam_hu=5.0, lam_bc=5.0, p=64, seed=0, log=2500):
    tf.keras.utils.set_random_seed(seed)
    mdl = FourBranchDeepONet(m=M, p=p, fusion=fusion, ic_mode="exp")
    opt = tf.keras.optimizers.Adam(
        tf.keras.optimizers.schedules.ExponentialDecay(1e-3, 10000, 0.5, staircase=True))

    NS, NX = len(TS), NXQ
    xq = tf.constant(np.tile(XQ, NS)[:, None])
    tq = tf.constant(np.repeat(TS, NX)[:, None])
    h0q = tf.constant(np.tile(H0[:N_SUP], (1, NS)).astype(np.float32))   # (N,NS*NX)
    bq  = tf.constant(np.tile(BB[:N_SUP], (1, NS)).astype(np.float32))
    href = tf.constant(H_snap.reshape(N_SUP, -1).astype(np.float32))
    huref = tf.constant(HU_snap.reshape(N_SUP, -1).astype(np.float32))
    h0s_t, bs_t = tf.constant(H0S), tf.constant(BS)

    def fwd(idx, xx, tt, hq, bqq):
        n = tf.shape(idx)[0]; nq = tf.shape(xx)[0]
        h0s = tf.repeat(tf.gather(h0s_t, idx), nq, axis=0)
        bs  = tf.repeat(tf.gather(bs_t,  idx), nq, axis=0)
        xr = tf.tile(xx, [n, 1]); tr = tf.tile(tt, [n, 1])
        return mdl([h0s, bs, tf.reshape(hq, [-1, 1]), tf.reshape(bqq, [-1, 1]), xr, tr])

    @tf.function(reduce_retracing=True)
    def step(idx, tbc):
        with tf.GradientTape() as tape:
            hh, hhu = fwd(idx, xq, tq, tf.gather(h0q, idx), tf.gather(bq, idx))
            hh = tf.reshape(hh, [tf.shape(idx)[0], -1])
            hhu = tf.reshape(hhu, [tf.shape(idx)[0], -1])
            Ld = tf.reduce_mean((hh - tf.gather(href, idx)) ** 2) \
                 + lam_hu * tf.reduce_mean((hhu - tf.gather(huref, idx)) ** 2)
            # periodic BC at x=0 and x=L
            x0 = tf.zeros_like(tbc); xL = tf.fill(tf.shape(tbc), tf.constant(L, tf.float32))
            h0_0 = tf.gather(h0s_t, idx)[:, :1]; b_0 = tf.gather(bs_t, idx)[:, :1]
            hA, huA = fwd(idx, x0, tbc, tf.tile(h0_0, [1, tf.shape(tbc)[0]]),
                          tf.tile(b_0, [1, tf.shape(tbc)[0]]))
            hB, huB = fwd(idx, xL, tbc, tf.tile(h0_0, [1, tf.shape(tbc)[0]]),
                          tf.tile(b_0, [1, tf.shape(tbc)[0]]))
            Lb = tf.reduce_mean((hA - hB) ** 2) + tf.reduce_mean((huA - huB) ** 2)
            Ltot = Ld + lam_bc * Lb
        g = tape.gradient(Ltot, mdl.trainable_variables)
        g, _ = tf.clip_by_global_norm(g, 1.0)
        opt.apply_gradients(zip(g, mdl.trainable_variables))
        return Ld, Lb

    rng = np.random.default_rng(seed)
    t0 = time.time()
    for k in range(steps + 1):
        idx = tf.constant(rng.choice(N_SUP, 8, replace=False).astype(np.int32))
        tbc = tf.constant(rng.uniform(0, T, (64, 1)).astype(np.float32))
        Ld, Lb = step(idx, tbc)
        if k % log == 0:
            print(f"  [{fusion}] step {k:6d}  L_data {float(Ld):.3e}  L_bc {float(Lb):.3e}"
                  f"  ({time.time()-t0:.0f}s)")
    return mdl

MODELS = {f: train_variant(f, steps=15000) for f in ("add", "concat", "bilinear")}

## 4. Strong-bump case C2b — where additive fusion should break

C2 uses a 0.2 m bump on 1 m of water (weak coupling). C2b uses 0.5 m, so the source
term $-gh\,\partial_x b$ genuinely depends on the *product* of the two inputs. If
`concat`/`bilinear` beat `add` on C2b but not on C2, that is direct evidence for the
architectural point and a clean new row in Table 4.

In [ ]:
def evaluate(mdl, h0_prof, b_prof, t_eval=T, nx=400):
    xq_ = cell_centers(L, nx).astype(np.float32)
    h0_ = np.interp(xq_, xg, h0_prof, period=L).astype(np.float32)
    b_  = np.interp(xq_, xg, b_prof,  period=L).astype(np.float32)
    ref, _ = swe_solve(xq_.astype(float), h0_.astype(float), b_.astype(float),
                       float(t_eval), cfl=0.45, order=2)
    h0s = np.interp(xs, xq_, h0_, period=L).astype(np.float32)[None, :]
    bs  = np.interp(xs, xq_, b_,  period=L).astype(np.float32)[None, :]
    n = nx
    hp, hup = mdl([tf.tile(tf.constant(h0s), [n, 1]), tf.tile(tf.constant(bs), [n, 1]),
                   tf.constant(h0_)[:, None], tf.constant(b_)[:, None],
                   tf.constant(xq_)[:, None], tf.fill((n, 1), np.float32(t_eval))])
    hp = hp.numpy().ravel(); hup = hup.numpy().ravel()
    return (error_triplet(hp, ref[0], h_rest=float(ref[0].mean())),
            error_triplet(hup, ref[1], h_rest=0.0))

xq_ = cell_centers(L, 400)
CASES_EVAL = {
    "C1  flat bed":        (1 + 0.5*np.exp(-2*(xq_-5)**2), np.zeros_like(xq_)),
    "C2  bump 0.2 m":      (1 + 0.5*np.exp(-2*(xq_-5)**2), 0.2*np.exp(-(xq_-5)**2)),
    "C2b bump 0.5 m (NEW)":(1.4 + 0.5*np.exp(-2*(xq_-5)**2), 0.5*np.exp(-(xq_-5)**2)),
}
print(f"{'case':<24}{'fusion':<10}{'eps_h(tot)':>12}{'eps_h(anom)':>13}{'RMSE_h[m]':>12}{'eps_hu(tot)':>13}")
for cname, (h0p, bp) in CASES_EVAL.items():
    for f, mdl in MODELS.items():
        eh, ehu = evaluate(mdl, np.interp(xg, xq_, h0p, period=L),
                           np.interp(xg, xq_, bp, period=L))
        print(f"{cname:<24}{f:<10}{eh['rel_total']:>12.3e}{eh['rel_anomaly']:>13.3e}"
              f"{eh['rmse_m']:>12.3e}{ehu['rel_total']:>13.3e}")

## 5. A like-for-like speedup benchmark

Table 5 currently times a Python-loop CPU solver at **13× more timesteps than it
needs** against a batched GPU network. Two confounds, both inflating the number.

Fix all three legs:
- **CPU serial** — reference solver, one trajectory at a time (what a practitioner does today).
- **CPU ensemble-vectorised** — same solver, batched over the ensemble (fair software baseline).
- **GPU operator** — batched forward pass, timed after warm-up, with `.numpy()` sync.

Report all three. An honest $10^2$–$10^3\times$ is a good result; an unfalsifiable
$10^4\times$ is a referee magnet.

In [ ]:
def time_solver_serial(n, nx=400, reps=1):
    x_ = cell_centers(L, nx)
    rng_ = np.random.default_rng(1)
    h0_ = 1 + 0.3*np.sin(2*np.pi*(x_[None,:]+rng_.uniform(0,L,(n,1)))/L)
    t0 = time.perf_counter()
    for r in range(reps):
        for j in range(n):
            swe_solve(x_, h0_[j], np.zeros(nx), T, cfl=0.45, order=2)
    return (time.perf_counter() - t0) / (reps * n) * 1e3      # ms per trajectory

def time_solver_batch(n, nx=400, reps=1):
    x_ = cell_centers(L, nx)
    rng_ = np.random.default_rng(1)
    h0_ = 1 + 0.3*np.sin(2*np.pi*(x_[None,:]+rng_.uniform(0,L,(n,1)))/L)
    B_ = np.zeros((n, nx))
    t0 = time.perf_counter()
    for r in range(reps):
        swe_solve(x_, h0_, B_, T, cfl=0.45, order=2)
    return (time.perf_counter() - t0) / (reps * n) * 1e3

def time_operator(mdl, n, nx=400, reps=5):
    x_ = cell_centers(L, nx).astype(np.float32)
    h0s = tf.constant(np.tile(np.interp(xs, x_, 1+0.3*np.sin(2*np.pi*x_/L),
                                        period=L).astype(np.float32), (n*nx, 1)))
    bs = tf.zeros_like(h0s)
    h0q = tf.constant(np.tile((1+0.3*np.sin(2*np.pi*x_/L)).astype(np.float32), n)[:, None])
    bq = tf.zeros_like(h0q)
    xq2 = tf.constant(np.tile(x_, n)[:, None]); tq2 = tf.fill(tf.shape(xq2), np.float32(T))
    _ = mdl([h0s, bs, h0q, bq, xq2, tq2])                     # warm-up / trace
    t0 = time.perf_counter()
    for r in range(reps):
        h, hu = mdl([h0s, bs, h0q, bq, xq2, tq2]); h.numpy()  # force sync
    return (time.perf_counter() - t0) / (reps * n) * 1e3

mdl = MODELS["add"]
print(f"{'batch':>7}{'solver serial':>16}{'solver batched':>16}{'operator':>12}"
      f"{'speedup vs serial':>19}{'speedup vs batched':>20}")
for n in (1, 10, 100):
    ts, tb, to = time_solver_serial(n), time_solver_batch(n), time_operator(mdl, n)
    print(f"{n:>7}{ts:>16.2f}{tb:>16.2f}{to:>12.4f}{ts/to:>19.0f}{tb/to:>20.0f}")
print("\nAll times in ms per trajectory. State the CPU and GPU model in the caption,")
print("and note that the solver leg is single-threaded numpy.")

### Text changes this notebook supports

- Table 3 / abstract — add anomaly-normalised errors and dimensional RMSE; add
  $\bar\varepsilon_{hu}$ to the abstract; delete "capturing over 98.8% of the spatial variance".
- §3.4.2 — delete the claim that the trunk mediates the $h_0$–$b$ interaction; state that
  additive fusion is separable and report the C2b ablation.
- Table 4 — add fusion variants as rows; report the true 10k-step numbers for A3 rather
  than repeating the 40k values.
- Table 5 / §4.9 — replace with the three-leg benchmark; reconcile every number in the
  prose against the table (the current text disagrees with it in seven places).
- §4 — add the operator conservation figure.